In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [36]:
df_finaly = pd.read_csv(r"saved_data/df_finaly.csv")

In [37]:
df_finaly = df_finaly.drop(93,axis=0)
df_finaly = df_finaly.drop("№",axis=1)
df_finaly = df_finaly.drop("Стадия",axis=1)
df_finaly = df_finaly.drop("ФИО", axis = 1)
df_finaly.loc[95,"Пол"] = "ж"

In [38]:
# 1. Находим базовые названия колонок, у которых есть дубликаты _x и _y
cols_to_fix = set([c[:-2] for c in df_finaly.columns if c.endswith('_x') or c.endswith('_y')])

for col in cols_to_fix:
    col_x = f"{col}_x"
    col_y = f"{col}_y"
    
    # Если оба столбца существуют, объединяем их
    if col_x in df_finaly.columns and col_y in df_finaly.columns:
        # fillna объединит данные: если в _x пусто, возьмет из _y
        df_finaly[col] = df_finaly[col_x].fillna(df_finaly[col_y])
        # Удаляем временные столбцы с суффиксами
        df_finaly.drop([col_x, col_y], axis=1, inplace=True)
    # Если есть еще и столбец без суффикса (как ТГ), объединяем и с ним
    if col in df_finaly.columns and col + "_orig" not in df_finaly.columns: # техническая проверка
        pass

In [39]:
# @title
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder
from category_encoders import LeaveOneOutEncoder
class coding_methods():
    def __init__(self,df):
        self.df = df
    def OneHot_Encoding(self,column,handle_unknown = "error",drop = None,sparse = True,dtype = float):
        #каждую категорию превращаем в отдельную колонку с 0/1.
        ohe = OneHotEncoder(
        handle_unknown=handle_unknown,
        drop=drop,
        sparse_output=sparse,
        dtype=dtype
        )

        # Параметры, которые важны:
        # handle_unknown
        # Что делать, если при предсказании встречается новая категория.
        # — 'error': выбросит ошибку.
        # — 'ignore': создаст строку из нулей для незнакомого значения.
        # drop
        # Можно удалить одну категорию как базовую (drop='first'), чтобы уменьшить коллинеарность для линейных моделей.
        # sparse
        # True — возвращает разреженную матрицу, экономит память. False — numpy-массив.
        # dtype
        # Тип данных для выходных признаков.
        # OHE хорошо подходит деревьям и линейным моделям, но раздувает размерность, если категорий много.
        encoded = ohe.fit_transform(self.df[[column]])
        new_cols = ohe.get_feature_names_out([column])
        df_encoded = pd.DataFrame(encoded.toarray(), columns=new_cols)
        self.df = pd.concat([self.df.drop(columns=[column]), df_encoded], axis=1)
        return self.df

    def Ordinal_Encoding(self,column):
        #категория превращается в число 0, 1, 2, 3, …
        #Но модель может подумать, что между значениями есть расстояние, которое на самом деле отсутствует.
        enc = OrdinalEncoder(
        handle_unknown='error',
        unknown_value=None,
        encoded_missing_value=-1
        )
        # Параметры:

        # handle_unknown
        # Если встретится категория, которой не было при fit.
        # — 'error'
        # — 'use_encoded_value' (и тогда задаётся unknown_value).

        # unknown_value
        # Значение, в которое будут кодироваться неизвестные категории.

        # encoded_missing_value
        # Чем кодировать пропуски.

        # подходит деревьям решений, которые не реагируют на «порядок» категорий
        encoded = enc.fit_transform(self.df[[column]])
        encoded_data_shifted = encoded + 1
        self.df[column] = encoded_data_shifted
        return self.df
    def Target_Encoding(self,categorical_column,y_column):
        # Каждая категория заменяется средним значением целевой переменной для этой категории.
        # Категория сама как бы «предсказывает», насколько она близка к классу.

        # - Для классификации: p = mean(y).
        # - Для регрессии: тоже mean(y) по категории.

        # Часто используется библиотека category_encoders:
        enc = TargetEncoder(
        smoothing=1.0,
        min_samples_leaf=1
        )
        #Параметры:

        # - smoothing
        # Смягчение эффекта малых категорий: смесь глобального среднего и среднего в категории.
        # Большая smoothing ближе к глобальному mean.

        # - min_samples_leaf
        # Если в категории мало примеров, затираем статистику.

        # Target encoding чувствителен к утечкам, поэтому часто используют кросс-валидационную схему при обучении.
        self.df[categorical_column] = enc.fit_transform(self.df[categorical_column], self.df[y_column])
        return self.df
    def Frequency_or_Count_Encoding(self,column):
        # Кодируем категорию частотой (отношением числа строк класса) или простым количеством.
        # Модель получает информацию о «популярности» значения.
        self.df[column] = self.df[column].map(self.df[column].value_counts())
        return self.df
    # def Leave_One_Out_Encoding(self,column,sigma = 0.1):
    #     enc = LeaveOneOutEncoder(
    #         sigma=sigma
    #     )

    #     self.df[column] = enc.fit_transform(self.df[column])
    #     return self.df

a = coding_methods(df_finaly)
# z = coding_methods()

In [40]:
# df_finaly = df_finaly.drop("Форма",axis=1)

In [41]:
df_finaly = a.Ordinal_Encoding("Форма")
df_finaly = a.Ordinal_Encoding("Пол")
df_finaly
df_finaly.info()

<class 'pandas.DataFrame'>
Index: 95 entries, 0 to 95
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ПОЛ в плазме              95 non-null     float64
 1   ПОЛ в мембр. Эритроцитов  95 non-null     float64
 2   Лактат                    84 non-null     float64
 3   Глюкоза                   84 non-null     float64
 4   Общий белок               84 non-null     float64
 5   Мочевина                  84 non-null     float64
 6   Кортизол                  84 non-null     float64
 7   Зонулин                   84 non-null     float64
 8   Холестерин                84 non-null     float64
 9   здоров(1)\болен(0)        95 non-null     float64
 10  ЭХС                       84 non-null     float64
 11  НЭЖК                      84 non-null     float64
 12  Пол                       95 non-null     float64
 13  Возраст                   63 non-null     float64
 14  Форма                     95

In [42]:
mean_df_without_spliting_data = df_finaly.fillna(df_finaly.mean(), inplace=False)
mean_df_without_spliting_data = mean_df_without_spliting_data.drop("Форма",axis=1)
mean_df_without_spliting_data.to_csv("saved_data/mean_df_without_spliting_data.csv", index = False)

In [43]:
# # @title
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from scipy import stats

# # Функции для корреляций / ассоциаций

# def cramers_v(x, y):
#     contingency = pd.crosstab(x, y)
#     chi2, p, dof, expected = stats.chi2_contingency(contingency)
#     n = contingency.sum().sum()
#     min_dim = min(contingency.shape) - 1
#     if min_dim == 0:
#         return np.nan
#     return np.sqrt(chi2 / (n * min_dim))


# def point_biserial_correlation(numeric, binary):
#     # убедимся, что не передаём NaN
#     # .correlation возвращает float or nan
#     return stats.pointbiserialr(numeric, binary).correlation


# def analyze_column(series: pd.Series, col_name: str):
#     arr = series.dropna().values
#     if arr.size == 0:
#         print(f"Столбец '{col_name}' пуст после удаления NaN — пропускаем.")
#         return

#     mean_val = np.mean(arr)
#     median_val = np.median(arr)

#     mode_res = stats.mode(arr, keepdims=False)
#     m = mode_res.mode
#     if isinstance(m, np.ndarray):
#         mode_val = m[0] if m.size > 0 else np.nan
#     else:
#         mode_val = float(m)

#     var_sample = np.var(arr, ddof=1)
#     stddev = np.std(arr, ddof=1)
#     q1 = np.quantile(arr, 0.25)
#     q3 = np.quantile(arr, 0.75)
#     iqr = q3 - q1

#     print(f"\n=== Статистика для столбца '{col_name}' ===")
#     print(f"  Среднее (mean): {mean_val}")
#     print(f"  Медиана (median): {median_val}")
#     print(f"  Мода (mode): {mode_val}")
#     print(f"  Выборочная дисперсия: {var_sample}")
#     print(f"  Стандартное отклонение: {stddev}")
#     print(f"  Q1: {q1}, Q3: {q3}, IQR: {iqr}")

#     sns.set(style="whitegrid")
#     plt.figure(figsize=(12, 8))

#     # 1. KDE + среднее, медиана, мода
#     ax1 = plt.subplot(2, 2, 1)
#     sns.kdeplot(series.dropna(), fill=True, color='skyblue', label='KDE')
#     ax1.axvline(mean_val, color='red', linestyle='--', label=f"Mean = {mean_val:.2f}")
#     ax1.axvline(median_val, color='green', linestyle='-.', label=f"Median = {median_val:.2f}")
#     ax1.axvline(mode_val, color='purple', linestyle=':', label=f"Mode = {mode_val:.2f}")
#     ax1.legend()
#     ax1.set_title(f"KDE + mean/median/mode — {col_name}")

#     # 2. Гистограмма + KDE
#     ax2 = plt.subplot(2, 2, 2)
#     sns.histplot(series.dropna(), bins=30, kde=True, color='lightgray')
#     ax2.set_title(f"Гистограмма + KDE — {col_name}")

#     # 3. Boxplot (ящики с усиками)
#     ax3 = plt.subplot(2, 2, 3)
#     sns.boxplot(x=series.dropna(), color='lightgreen')
#     ax3.set_title(f"Boxplot — {col_name}")

#     # 4. Накопительная дисперсия
#     ax4 = plt.subplot(2, 2, 4)
#     subs = np.arange(10, len(arr) + 1, step=max(1, len(arr)//10))
#     var_vals = []
#     for k in subs:
#         subarr = arr[:k]
#         if subarr.size > 1:
#             var_vals.append(np.var(subarr, ddof=1))
#         else:
#             var_vals.append(np.nan)
#     ax4.plot(subs, var_vals, marker='o')
#     ax4.set_xlabel("Количество первых точек (k)")
#     ax4.set_ylabel("Выборочная дисперсия")
#     ax4.set_title(f"Накопительная дисперсия — {col_name}")

#     plt.tight_layout()
#     plt.show()


# def analyze_dataframe_all_corr(df: pd.DataFrame, method_numeric: str = "pearson"):
#     cols = df.columns.tolist()
#     corr_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

#     numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
#     categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

#     for i in cols:
#         for j in cols:
#             xi = df[i]
#             xj = df[j]

#             if i == j:
#                 corr_matrix.at[i, j] = 1.0
#             else:
#                 if i in numeric_cols and j in numeric_cols:
#                     corr = xi.corr(xj, method=method_numeric)
#                     corr_matrix.at[i, j] = corr
#                 elif i in categorical_cols and j in categorical_cols:
#                     try:
#                         cv = cramers_v(xi, xj)
#                     except Exception:
#                         cv = np.nan
#                     corr_matrix.at[i, j] = cv
#                 else:
#                     # смешанный случай
#                     # i — числовой, j — категориальный
#                     if i in numeric_cols and j in categorical_cols:
#                         uniques = xj.dropna().unique()
#                         if len(uniques) == 2:
#                             mapping = {u: idx for idx, u in enumerate(uniques)}
#                             bin_col = xj.map(mapping)
#                             # выберем только те индексы, где numeric не NaN
#                             valid_idx = xi.dropna().index
#                             corr = point_biserial_correlation(xi.loc[valid_idx], bin_col.loc[valid_idx])
#                             corr_matrix.at[i, j] = corr
#                         else:
#                             groups = []
#                             for cat in uniques:
#                                 group = xi[xj == cat].dropna().values
#                                 if len(group) > 0:
#                                     groups.append(group)
#                             try:
#                                 f_stat, p_val = stats.f_oneway(*groups)
#                                 n = xi.dropna().shape[0]
#                                 k = len(groups)
#                                 eta2 = (f_stat * (n - k)) / (f_stat * (n - k) + (k - 1))
#                             except Exception:
#                                 eta2 = np.nan
#                             corr_matrix.at[i, j] = eta2
#                     else:
#                         # i — категориальный, j — числовой
#                         corr_matrix.at[i, j] = corr_matrix.at[j, i]

#     print("Матрица ассоциаций:")
#     print(corr_matrix)
#     plt.figure(figsize=(max(8, len(cols)*0.5), max(6, len(cols)*0.4)))
#     sns.heatmap(corr_matrix.astype(float), annot=True, fmt=".2f", cmap="coolwarm", center=0)
#     plt.title("Матрица ассоциаций между признаками")
#     plt.show()

#     return corr_matrix


# def plot_boxplots_all_with_median_color(df: pd.DataFrame):
#     """
#     Построение Boxplot для всех колонок с цветом по медиане.
#     Для категориальных колонок значения кодируются числами.
#     """
#     df_plot = df.copy()

#     # Преобразуем категориальные колонки в числовые
#     for col in df_plot.columns:
#         if df_plot[col].dtype == 'object' or str(df_plot[col].dtype).startswith('category'):
#             df_plot[col] = df_plot[col].astype('category').cat.codes

#     # Вычисляем медианы для всех колонок
#     medians = df_plot.median()
#     # Нормализуем медианы для палитры
#     norm = (medians - medians.min()) / (medians.max() - medians.min())
#     colors = sns.color_palette("coolwarm", n_colors=len(norm))
#     palette = [colors[int(v*(len(colors)-1))] for v in norm]

#     plt.figure(figsize=(max(8, len(df_plot.columns)*0.7), 6))
#     sns.boxplot(data=df_plot, palette=palette)
#     plt.title("Boxplot всех признаков (цвет по медиане)")
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()

# def plot_scatter_all_numeric(df: pd.DataFrame):
#     """
#     Построение диаграмм разброса (scatter) для всех пар числовых колонок.
#     """
#     numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
#     if len(numeric_cols) < 2:
#         print("Недостаточно числовых колонок для построения scatter plot.")
#         return

#     # Создаем сетку графиков для всех пар
#     n = len(numeric_cols)
#     plt.figure(figsize=(5*n, 5*n))

#     plot_idx = 1
#     for i in range(n):
#         for j in range(i+1, n):
#             plt.subplot(n-1, n-1, plot_idx)
#             plt.scatter(df[numeric_cols[i]], df[numeric_cols[j]], alpha=0.6)
#             plt.xlabel(numeric_cols[i])
#             plt.ylabel(numeric_cols[j])
#             plt.title(f"{numeric_cols[i]} vs {numeric_cols[j]}")
#             plot_idx += 1

#     plt.tight_layout()
#     plt.show()

# def analyze_dataframe(df: pd.DataFrame):
#     numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

#     if numeric_cols:
#         for col in numeric_cols:
#             analyze_column(df[col], col)

#     # Общая матрица корреляций / ассоциаций
#     analyze_dataframe_all_corr(df)
#     # # Отдельный общий boxplot для всех числовых колонок
#     # plot_boxplots_all_with_median_color(df)
#     # # Диаграмма разброса всех числовых колонок
#     # plot_scatter_all_numeric(df)

# if __name__ == "__main__":
#     # df = pd.read_csv("UberDataset.csv")
#     # df1 = pd.read_csv("ecommerce_customer_data_custom_ratios.csv")
#     # df2 = pd.read_csv("ecommerce_customer_data_large.csv")
#     print(df.info())
#     print(df.describe())
#     print("Пропущенные значения:\n", df.isnull().sum())
#     analyze_dataframe(mean_df)
#     # analyze_dataframe(df1)
#     # analyze_dataframe(df2)
